# **Limpeza e Extração de Dados das Súmulas CBF**

## **1. Bibliotecas**

Além das dependências padrão, este notebook usa:
- `pdfplumber` — extração de texto de PDFs multi-página
- `unicodedata` — normalização de strings com acentos para matching robusto

In [14]:
import re
import unicodedata
from pathlib import Path

import pandas as pd
import pdfplumber

## **2. Configuração**

Carrega o índice de súmulas gerado no notebook 0. Cada linha corresponde a uma súmula baixada, com metadados de jogo e caminho do PDF.

Este notebook produz dois CSVs:

- `card_reasons.csv`: uma linha por cartão (amarelo ou vermelho);
- `delay_reasons.csv`: uma linha por partida com o motivo de atraso/acréscimos.

In [15]:
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
DATA_PATH = PROJECT_ROOT / "data"
RAW_PATH = DATA_PATH / "raw" / "match-reports"
PROCESSED_PATH = DATA_PATH / "processed"
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

INDEX_PATH = RAW_PATH / "match_reports_index.csv"
CARD_REASONS_PATH = PROCESSED_PATH / "card_reasons.csv"
DELAY_REASONS_PATH = PROCESSED_PATH / "delay_reasons.csv"
SUCCESS_DOWNLOAD_STATUSES = {"exists", "downloaded"}

index = pd.read_csv(INDEX_PATH)
index = index[index["download_status"].isin(SUCCESS_DOWNLOAD_STATUSES)].reset_index(drop=True)
index.head(2)

,competition,competition_name,year,competition_id,round,game_id,game_number,home_team,away_team,match_date,match_time,venue,document_title,document_title_normalized,document_url,file_path,download_status
0,copa-do-brasil,Copa do Brasil,2020,1361,1,814689,1,River,Esporte Clube Bahia,NaN,NaN,NaN,Súmula,sumula,https://conteudo.cbf.com.br/sumulas/2020/4241s...,/Users/gcarninatto/Documents/GitHub/cbf-match-...,downloaded
1,copa-do-brasil,Copa do Brasil,2020,1361,1,814698,2,Frei Paulistano,Remo,NaN,NaN,NaN,Súmula,sumula,https://conteudo.cbf.com.br/sumulas/2020/42410...,/Users/gcarninatto/Documents/GitHub/cbf-match-...,downloaded


## **3. Extração de Texto dos PDFs**

Cada súmula é um PDF de 2–3 páginas. O texto de todas as páginas é concatenado em uma única string, que serve de entrada para as funções de parsing por regex.

`resolve_pdf_path` resolve caminhos relativos registrados no índice independentemente do diretório de trabalho atual.

In [16]:
def resolve_pdf_path(file_path: str) -> Path:
    path = Path(file_path)
    if path.is_absolute():
        return path
    candidate = (PROJECT_ROOT / path).resolve()
    if candidate.exists():
        return candidate
    return (PROJECT_ROOT / "notebooks" / path).resolve()


def extract_pdf_text(file_path: str) -> str:
    with pdfplumber.open(resolve_pdf_path(file_path)) as pdf:
        return "\n".join(page.extract_text() or "" for page in pdf.pages)

## **4. Funções de Parsing**

### **4.1 Identificação de Times**

O matching de nomes de times usa normalização sem acento e minúsculo para lidar com variações entre o índice e o texto do PDF — por exemplo, `"Fortaleza Esporte Clube"` no índice vs. `"Fortaleza/CE"` no PDF. A pontuação (`score`) favorece correspondências no final da linha, onde o nome do time geralmente aparece nos cabeçalhos de cartão.

In [17]:
def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFKD", str(text))
    text = "".join(c for c in text if not unicodedata.combining(c))
    return text.casefold().strip()


def _prep_for_match(text: str) -> str:
    n = normalize_text(text)
    n = re.sub(r"/[a-z]{2,3}\s*$", "", n).rstrip()
    return re.sub(r"[^\w\s]", "", n).strip()


def identify_team(line: str, home_team: str, away_team: str) -> str | None:
    line_clean = _prep_for_match(line)
    home_n = re.sub(r"[^\w\s]", "", normalize_text(home_team)).strip()
    away_n = re.sub(r"[^\w\s]", "", normalize_text(away_team)).strip()

    def score(team_n: str, line_c: str) -> float:
        if line_c.endswith(team_n):
            return 3.0
        tokens = [t for t in team_n.split() if len(t) > 3]
        last3 = " ".join(line_c.split()[-3:])
        for tok in tokens:
            if line_c.endswith(tok):
                return 1.0
        for tok in tokens:
            if tok in last3:
                return 0.5
        return 0.0

    hs = score(home_n, line_clean)
    as_ = score(away_n, line_clean)
    if hs > as_:
        return home_team
    if as_ > hs:
        return away_team
    return None

### **4.2 Campos Simples**

Extrai árbitro e data diretamente via regex. A data é convertida de `DD/MM/AAAA` para o formato ISO `AAAA-MM-DD`.

In [18]:
def parse_arbitro(text: str) -> str | None:
    m = re.search(r"Arbitro:\s*([^(\n]+)", text)
    return m.group(1).strip() if m else None


def parse_match_date(match_date_raw: str) -> str | None:
    dt = pd.to_datetime(str(match_date_raw).strip(), format="%d/%m/%Y", errors="coerce")
    return dt.strftime("%Y-%m-%d") if not pd.isnull(dt) else None

### 4.3 Parsing de Cartões

Cada entrada de cartão no PDF tem a estrutura:

```text
Motivo: <código e texto padronizado>  -  <justificativa escrita pelo árbitro>
```

Este notebook salva somente a justificativa escrita pelo árbitro, ou seja, a parte depois do separador ` - `. Todo o trecho anterior ao separador é descartado.

In [19]:
TIMESTAMP_RE = re.compile(r"^(?P<minute>\d{2}):(?P<second>\d{2})\s+(?P<half>[12]T)\s+")
SKIP_RE = re.compile(r"^(Tempo 1T|NÃO HOUVE|[A-Z]{2,3} -|Confederação|\d+\s*/\s*\d+)")


def clean_motivo(raw: str) -> str:
    cleaned = re.sub(r"^[A-Z]\d*(?:\.\d+)*\.?\s*", "", raw.strip())
    return re.sub(r"\s+", " ", cleaned).strip()


def normalize_reason(text: str) -> str:
    text = normalize_text(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def extract_referee_justification(raw_reason: str) -> str:
    """Retorna somente a justificativa livre do árbitro após o separador."""
    cleaned = clean_motivo(raw_reason)
    parts = re.split(r"\s+[–-]\s*", cleaned, maxsplit=1)
    if len(parts) < 2:
        return ""
    justification = re.sub(r"^[-–]\s*", "", parts[1]).strip()
    return re.sub(r"\s+", " ", justification).strip()


def get_section(text: str, start_marker: str, end_marker: str) -> str:
    m = re.search(
        rf"{re.escape(start_marker)}\n(.+?)(?={re.escape(end_marker)})",
        text,
        re.DOTALL,
    )
    return m.group(1) if m else ""


def parse_card_section(
    section_text: str,
    home_team: str,
    away_team: str,
    card_type: str,
) -> list[dict]:
    cards: list[dict] = []
    current_card: dict | None = None
    motivo_lines: list[str] = []
    in_motivo = False

    def flush_current() -> None:
        if current_card and motivo_lines:
            reason = extract_referee_justification(" ".join(motivo_lines))
            if reason:
                cards.append({
                    **current_card,
                    "reason": reason,
                    "reason_key": normalize_reason(reason),
                })

    for line in section_text.split("\n"):
        line = line.strip()
        if not line or SKIP_RE.match(line):
            continue

        timestamp = TIMESTAMP_RE.match(line)
        if timestamp:
            flush_current()
            current_card = {
                "card_type": card_type,
                "team": identify_team(line, home_team, away_team),
                "minute": int(timestamp.group("minute")),
                "half": timestamp.group("half"),
            }
            motivo_lines = []
            in_motivo = False

        elif "Motivo:" in line:
            in_motivo = True
            raw = re.sub(r"^.*?Motivo:\s*", "", line)
            motivo_lines = [raw] if raw else []

        elif in_motivo and not SKIP_RE.match(line):
            motivo_lines.append(line)

    flush_current()
    return [card for card in cards if card.get("team") and card.get("reason")]


### 4.4 Motivo de Atraso / Acréscimos

Extrai o campo "Motivo de atraso no início e/ou reinício, e de acréscimos" da seção de Ocorrências. Se a seção estiver ausente, retorna `"n/a"`.

In [20]:
_DELAY_STOP_RE = re.compile(
    r"^(Observa[çc]|Relat[oó]rio do Assistente|Substitui[çc])",
    re.IGNORECASE,
)


def parse_delay_reason(text: str) -> str:
    lines = text.split("\n")
    collecting = False
    collected: list[str] = []

    for line in lines:
        stripped = line.strip()
        if re.search(r"Motivo de atraso", stripped, re.IGNORECASE):
            collecting = True
            after = re.sub(r".*?:\s*", "", stripped, count=1).strip()
            if after:
                collected.append(after)
            continue
        if collecting:
            if _DELAY_STOP_RE.match(stripped):
                break
            if stripped:
                collected.append(stripped)

    if not collected:
        return "n/a"
    return re.sub(r"\s+", " ", " ".join(collected)).strip()

## **5. Construção dos Datasets**

Duas bases geradas por partida:

- `card_reasons`: uma linha por cartão, com `card_type`, `minute`, `half`, `team` e `reason`;
- `delay_reasons`: uma linha por partida com o motivo de atraso/acréscimos escrito pelo árbitro.

Em `card_reasons`, `reason` é sempre a justificativa livre do árbitro após o separador ` - ` no campo `Motivo`.

In [21]:
def process_match(row: pd.Series, text: str) -> tuple[list[dict], dict]:
    home_team = row["home_team"]
    away_team = row["away_team"]

    yellow_section = get_section(text, "Cartões Amarelos", "Cartões Vermelhos")
    red_section = get_section(text, "Cartões Vermelhos", "Ocorrências / Observações")

    yellow_cards = parse_card_section(yellow_section, home_team, away_team, "yellow")
    red_cards = parse_card_section(red_section, home_team, away_team, "red")

    arbitro = parse_arbitro(text)
    match_date = parse_match_date(row["match_date"])

    card_records = []
    for card in yellow_cards + red_cards:
        card_records.append({
            "year": row["year"],
            "round": row["round"],
            "game_id": row["game_id"],
            "game_number": row["game_number"],
            "match_date": match_date,
            "home_team": home_team,
            "away_team": away_team,
            "arbitro": arbitro,
            **card,
        })

    delay_record = {
        "year": row["year"],
        "round": row["round"],
        "game_id": row["game_id"],
        "game_number": row["game_number"],
        "match_date": match_date,
        "home_team": home_team,
        "away_team": away_team,
        "arbitro": arbitro,
        "delay_reason": parse_delay_reason(text),
    }

    return card_records, delay_record

In [22]:
all_card_records = []
all_delay_records = []

for _, row in index.iterrows():
    text = extract_pdf_text(row["file_path"])
    card_records, delay_record = process_match(row, text)
    all_card_records.extend(card_records)
    all_delay_records.append(delay_record)

In [23]:
card_reason_cols = [
    "year",
    "round",
    "game_id",
    "game_number",
    "match_date",
    "home_team",
    "away_team",
    "arbitro",
    "card_type",
    "team",
    "minute",
    "half",
    "reason",
    "reason_key",
]
delay_reason_cols = [
    "year",
    "round",
    "game_id",
    "game_number",
    "match_date",
    "home_team",
    "away_team",
    "arbitro",
    "delay_reason",
]

card_reasons = (
    pd.DataFrame(all_card_records)
    .reindex(columns=card_reason_cols)
    .sort_values(["year", "round", "game_number", "card_type", "half", "minute"])
    .reset_index(drop=True)
)
delay_reasons = (
    pd.DataFrame(all_delay_records)
    .reindex(columns=delay_reason_cols)
    .sort_values(["year", "round", "game_number"])
    .reset_index(drop=True)
)

card_reasons.head()

,year,round,game_id,game_number,match_date,home_team,away_team,arbitro,card_type,team,minute,half,reason,reason_key
0,2020,1,812497,1,2019-04-27,Guarani,Figueirense,Luiz César de Oliveira Magalhães,yellow,Figueirense,7,1T,Por dar uma rasteira em seu adversário de mane...,por dar uma rasteira em seu adversario de mane...
1,2020,1,812497,1,2019-04-27,Guarani,Figueirense,Luiz César de Oliveira Magalhães,yellow,Guarani,15,1T,Por dar uma rasteira em seu adversário de mane...,por dar uma rasteira em seu adversario de mane...
2,2020,1,815072,1,2020-08-09,Flamengo,Atlético Mineiro,Raphael Claus,yellow,Atlético Mineiro,18,1T,Por reclamar ostensivamente com palavras e ges...,por reclamar ostensivamente com palavras e ges...
3,2020,1,812497,1,2019-04-27,Guarani,Figueirense,Luiz César de Oliveira Magalhães,yellow,Figueirense,25,1T,Por dar um pontapé (chute) em seu adversário d...,por dar um pontape (chute) em seu adversario d...
4,2020,1,814689,1,None,River,Esporte Clube Bahia,Jefferson Ferreira de Moraes,yellow,River,35,1T,Por impedir um ataque promissor.,por impedir um ataque promissor.


## **6. Validação**

Verificação rápida de cobertura: total de cartões extraídos, total de partidas com motivo de atraso preenchido vs. `"n/a"`, e número de textos únicos no campo `reason` — proxy da diversidade de descrições escritas pelos árbitros.

In [24]:
validation_summary = pd.Series({
    "card_reason_rows": len(card_reasons),
    "delay_reason_rows": len(delay_reasons),
    "delay_na": (delay_reasons["delay_reason"] == "n/a").sum(),
    "unique_reason_keys": card_reasons["reason_key"].nunique(),
})
validation_summary

card_reason_rows      25756
delay_reason_rows      5663
delay_na                 52
unique_reason_keys     7607
dtype: int64

In [25]:
card_reasons.groupby(["year", "card_type"]).size().unstack(fill_value=0)

card_type,red,yellow
year,,
2020,197,3695
2021,170,3604
2022,180,3842
2023,174,3868
2024,187,4066
2025,184,3895
2026,87,1607


## **7. Salvamento**

Arquivos gerados:

- `data/processed/card_reasons.csv`: uma linha por cartão;
- `data/processed/delay_reasons.csv`: uma linha por partida com o motivo de atraso/acréscimos.

In [26]:
card_reasons.to_csv(CARD_REASONS_PATH, index=False)
delay_reasons.to_csv(DELAY_REASONS_PATH, index=False)
{
    "card_reasons": card_reasons.shape,
    "delay_reasons": delay_reasons.shape,
}

{'card_reasons': (25756, 14), 'delay_reasons': (5663, 9)}